# Best ABSA Model: PhoBERT BiLSTM-CRF

Notebook nay train lai model clean tot nhat hien tai cua repo: PhoBERT base v2 + character CNN + BiLSTM + CRF, sequence unit = token. Metric chinh van la exact character-span + label F1 tren dev.

## Install Dependencies

In [ ]:
!pip install -q transformers sentencepiece protobuf scikit-learn

## Configure Project

In [ ]:
import os
from datetime import datetime

REPO_URL = "https://github.com/ngocvo2511/NLP.git"
REPO_BRANCH = "best-phobert-bilstm-crf"
PROJECT_DIR = "/kaggle/working/NLP"
RUN_NAME = datetime.utcnow().strftime("best_phobert_bilstm_crf_%Y%m%d_%H%M%S")
OUTPUT_ROOT = f"/kaggle/working/NLP_outputs/{RUN_NAME}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["OUTPUT_ROOT"] = OUTPUT_ROOT

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
    !git pull origin {REPO_BRANCH}

%cd {PROJECT_DIR}
!python -m src.absa.validate_data --data-dir data
print("Output root:", OUTPUT_ROOT)
print("Branch:", REPO_BRANCH)

## Train Best Dev Model

Cau hinh nay la PhoBERT BiLSTM-CRF token-unit. Day la nhanh co dev exact F1 tot nhat da ghi nhan truoc do, khoang 0.5954.

In [ ]:
!python -m src.absa.train_transformer_bilstm_crf \
  --model-name vinai/phobert-base-v2 \
  --data-dir data \
  --output-dir "$OUTPUT_ROOT/phobert-bilstm-crf-best" \
  --tag-scheme bio \
  --hidden-size 512 \
  --layers 1 \
  --char-dim 64 \
  --char-out 128 \
  --dropout 0.3 \
  --batch-size 8 \
  --eval-batch-size 16 \
  --epochs 12 \
  --learning-rate 2e-5 \
  --head-learning-rate 1e-3 \
  --weight-decay 1e-5 \
  --seed 42

!cat "$OUTPUT_ROOT/phobert-bilstm-crf-best/dev_metrics.json"

## Optional Seed Repeats

In [ ]:
# for seed in [2024, 3407]:
#     name = f"phobert-bilstm-crf-best-seed-{seed}"
#     print("\n===", name, "===")
#     !python -m src.absa.train_transformer_bilstm_crf \
#       --model-name vinai/phobert-base-v2 \
#       --data-dir data \
#       --output-dir "$OUTPUT_ROOT/{name}" \
#       --tag-scheme bio \
#       --hidden-size 512 \
#       --layers 1 \
#       --char-dim 64 \
#       --char-out 128 \
#       --dropout 0.3 \
#       --batch-size 8 \
#       --eval-batch-size 16 \
#       --epochs 12 \
#       --learning-rate 2e-5 \
#       --head-learning-rate 1e-3 \
#       --weight-decay 1e-5 \
#       --seed {seed}
#     !cat "$OUTPUT_ROOT/{name}/dev_metrics.json"

## Export Compact Results

In [ ]:
import json
import shutil
from pathlib import Path

compact_dir = Path('/kaggle/working/absa_best_phobert_bilstm_crf_outputs')
compact_dir.mkdir(parents=True, exist_ok=True)

for path in Path(OUTPUT_ROOT).rglob('*'):
    if path.name in {'dev_metrics.json', 'dev_predictions.jsonl', 'history.json', 'metadata.json'}:
        target = compact_dir / path.relative_to(OUTPUT_ROOT)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)

archive = shutil.make_archive(str(compact_dir), 'zip', compact_dir)
print('Compact output directory:', compact_dir)
print('Compact zip:', archive)

for metrics_path in sorted(Path(OUTPUT_ROOT).glob('*/dev_metrics.json')):
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(metrics_path.parent.name, metrics['micro'])